In [51]:
# Step 1: Import necessary libraries and load the dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load the chosen dataset
df = pd.read_csv('activity_log.csv')
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

Dataset loaded with 689 rows and 17 columns.


In [52]:
# Step 2: Keep only running activities
df = df[df['Activity Type'].isin(['Running', 'Treadmill Running'])].copy()

# Function to convert time string (HH:MM:SS or MM:SS) to total minutes
def time_to_minutes(time_str):
    try:
        parts = str(time_str).split(':')
        if len(parts) == 3:
            return int(parts[0]) * 60 + int(parts[1]) + float(parts[2]) / 60
        elif len(parts) == 2:
            return int(parts[0]) + float(parts[1]) / 60
        else:
            return np.nan
    except:
        return np.nan

In [53]:
# Clean target variable (Time) and features
df['Time_Minutes'] = df['Time'].apply(time_to_minutes)
df['Avg_Pace_Min'] = df['Avg Pace'].apply(time_to_minutes)

# Clean numeric columns that might be stored as strings (e.g., '144', '--')
cols_to_numeric = ['Avg HR', 'Max HR', 'Avg Run Cadence', 'Calories']
for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [57]:
# Drop rows with critical missing values to ensure high-quality training data
df.dropna(subset=['Time_Minutes', 'Distance', 'Avg HR', 'Max HR', 'Avg Run Cadence'], inplace=True)

# Sort by date to calculate rolling features properly
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

print("Data cleaning complete. Ready for feature engineering.")

Data cleaning complete. Ready for feature engineering.


In [58]:
# Step 3: Calculate a 7-day rolling sum of distance to represent 'Weekly Mileage'
df.set_index('Date', inplace=True)
df['Weekly_Mileage'] = df['Distance'].rolling('7D').sum()
df.reset_index(inplace=True)

# THE FIX: Create the new target variable (Pace = Time / Distance)
df['Target_Pace'] = df['Time_Minutes'] / df['Distance']

In [40]:
# Define the features (X) and the NEW target variable (y)
features = ['Distance', 'Avg HR', 'Max HR', 'Avg Run Cadence', 'Weekly_Mileage']
target = 'Target_Pace'  # <--- Changed from Time_Minutes

X = df[features]
y = df[target]

print(f"Features selected: {features}")
print(f"Target variable: {target}")

Features selected: ['Distance', 'Avg HR', 'Max HR', 'Avg Run Cadence', 'Weekly_Mileage']
Target variable: Target_Pace


In [75]:
#Step 4: Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=24)

# Initialize models
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# Train models on PACE
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

print("Models successfully trained on Pace.")

Models successfully trained on Pace.


In [76]:
# Step 5: Evaluate the models using MAE, RMSE, and R^2 metrics
def evaluate_model(name, model, X_test, y_test):
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    
    print(f"--- {name} Evaluation (Pace) ---")
    print(f"Pace MAE:  {mae:.2f} minutes/unit")
    print(f"Pace RMSE: {rmse:.2f} minutes/unit")
    print(f"R^2:  {r2:.4f}\n")

In [77]:
evaluate_model("Random Forest", rf_model, X_test, y_test)
evaluate_model("XGBoost", xgb_model, X_test, y_test)

# Select the Champion Model
champion_model = xgb_model

--- Random Forest Evaluation (Pace) ---
Pace MAE:  0.27 minutes/unit
Pace RMSE: 0.37 minutes/unit
R^2:  0.8780

--- XGBoost Evaluation (Pace) ---
Pace MAE:  0.28 minutes/unit
Pace RMSE: 0.38 minutes/unit
R^2:  0.8725



In [78]:
def predict_race_time(target_distance, expected_avg_hr, max_hr, expected_cadence, current_weekly_mileage):
    """
    Predicts race finish time by first predicting PACE, then multiplying by DISTANCE.
    """
    input_data = pd.DataFrame({
        'Distance': [target_distance],
        'Avg HR': [expected_avg_hr],
        'Max HR': [max_hr],
        'Avg Run Cadence': [expected_cadence],
        'Weekly_Mileage': [current_weekly_mileage]
    })
    
    # 1. Model predicts your Pace (minutes per distance unit)
    predicted_pace = champion_model.predict(input_data)[0]
    
    # 2. Multiply Pace by Distance to get Total Minutes
    predicted_total_minutes = predicted_pace * target_distance
    
    # 3. Convert back to hours, minutes, seconds for human readability
    hours = int(predicted_total_minutes // 60)
    minutes = int(predicted_total_minutes % 60)
    seconds = int((predicted_total_minutes * 60) % 60)
    
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}", predicted_pace

# Example Predictions
distances = {
    "5K": 5.0,
    "10K": 10.0,
    "Half Marathon": 21.0975,
    "Full Marathon": 42.195
}

In [81]:
print("--- Personalized Race Time Predictions (Dynamic Race Effort) ---")

# Dynamically adjust the expected race HR and cadence based on the distance
# Calculated using a 208 Max HR baseline
race_efforts = {
    "5K": {"dist": 5.0, "hr": 196, "cadence": 180}, 
    "10K": {"dist": 10.0, "hr": 187, "cadence": 176}, 
    "Half Marathon": {"dist": 21.0975, "hr": 178, "cadence": 172}, 
    "Full Marathon": {"dist": 42.195, "hr": 170, "cadence": 168} 
}

for race, params in race_efforts.items():
    dist = params["dist"]
    race_hr = params["hr"]
    race_cadence = params["cadence"]
    
    # Inputting the dynamic race effort into the predictor
    pred_time, pred_pace = predict_race_time(
        target_distance=dist, 
        expected_avg_hr=race_hr, 
        max_hr=201, 
        expected_cadence=race_cadence, 
        current_weekly_mileage=65
    )
    
    # Format pace into MM:SS
    pace_min = int(pred_pace)
    pace_sec = int((pred_pace * 60) % 60)
    
    print(f"{race} ({dist:.2f} km) @ {race_hr} BPM")
    print(f"   Estimated Pace: {pace_min}:{pace_sec:02d} /km")
    print(f"   Estimated Time: {pred_time}\n")

--- Personalized Race Time Predictions (Dynamic Race Effort) ---
5K (5.00 km) @ 196 BPM
   Estimated Pace: 5:58 /km
   Estimated Time: 00:29:50

10K (10.00 km) @ 187 BPM
   Estimated Pace: 6:52 /km
   Estimated Time: 01:08:40

Half Marathon (21.10 km) @ 178 BPM
   Estimated Pace: 7:12 /km
   Estimated Time: 02:32:08

Full Marathon (42.20 km) @ 170 BPM
   Estimated Pace: 7:30 /km
   Estimated Time: 05:16:46

